# SofaScore Single-Match Odds Test

Enter any SofaScore match ID when prompted. The notebook opens the exact odds endpoint directly through undetected Chrome 150, extracts current full-time 1X2 decimal odds, and prints a structured result.

No direct HTTP client is used and no output file is created.


In [ ]:
# Imports
import json
from fractions import Fraction
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import undetected_chromedriver as uc
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Install the browser dependencies with "
        "'%pip install undetected-chromedriver selenium', then restart the kernel."
    ) from exc


In [ ]:
# Fixed browser settings
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 30
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 20


In [ ]:
# Enter the SofaScore match ID to test.
match_id_text = input("Enter a SofaScore match ID: ").strip()
# Handle expected failures with a clear, actionable message.
try:
    match_id = int(match_id_text)
except ValueError as exc:
    raise ValueError("The match ID must be a positive integer.") from exc

# Validate the input before continuing with later processing.
if match_id < 1:
    raise ValueError("The match ID must be a positive integer.")

print(f"Selected match ID: {match_id}")


In [ ]:
# Exceptions and fractional-to-decimal conversion
class OddsTestError(RuntimeError):
    pass


# Define Odds Retrieval Error to keep related behaviour explicit.
class OddsRetrievalError(OddsTestError):
    pass


# Define Odds Not Found Error to keep related behaviour explicit.
class OddsNotFoundError(OddsTestError):
    pass


# Define Invalid Odds JSONError to keep related behaviour explicit.
class InvalidOddsJSONError(OddsTestError):
    pass


# Define Odds Unavailable Error to keep related behaviour explicit.
class OddsUnavailableError(OddsTestError):
    pass


# Handle to decimal for reuse in the workflow.
def fractional_to_decimal(value: Any) -> float:
    # Validate the input before continuing with later processing.
    if not isinstance(value, str) or not value.strip():
        raise ValueError("fractionalValue must be a non-empty string.")
    # Handle expected failures with a clear, actionable message.
    try:
        fractional_odds = Fraction(value.strip())
    except (ValueError, ZeroDivisionError) as exc:
        raise ValueError(f"Invalid fractional odds value: {value!r}.") from exc
    # Validate the input before continuing with later processing.
    if fractional_odds < 0:
        raise ValueError(f"Fractional odds cannot be negative: {value!r}.")
    return round(float(fractional_odds + 1), 4)


In [ ]:
# Configure and initialize undetected Chrome 150.
options = uc.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

driver = None
# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(
        options=options,
        version_main=CHROME_MAJOR_VERSION,
        use_subprocess=True,
    )
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    driver.execute_cdp_cmd("Network.enable", {})
except Exception as exc:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
        except Exception:
            pass
        driver = None
    raise RuntimeError(
        "Could not initialize undetected Chrome 150. Confirm that a Chrome "
        f"150-compatible installation is available. Original error: {exc}"
    ) from exc

print("Undetected Chrome 150 initialized.")


In [ ]:
# Helpers for detecting the document HTTP status
def clear_performance_log(driver_instance: Any) -> None:
    # Handle expected failures with a clear, actionable message.
    try:
        driver_instance.get_log("performance")
    except Exception:
        pass


# Read performance log for reuse in the workflow.
def read_performance_log(driver_instance: Any) -> list[dict[str, Any]]:
    # Handle expected failures with a clear, actionable message.
    try:
        return driver_instance.get_log("performance")
    except Exception:
        return []


# Handle current url for reuse in the workflow.
def safe_current_url(driver_instance: Any) -> str | None:
    # Handle expected failures with a clear, actionable message.
    try:
        return driver_instance.current_url
    except Exception:
        return None


# Find document status for reuse in the workflow.
def find_document_status(
    log_entries: list[dict[str, Any]],
    requested_url: str,
    current_url: str | None = None,
) -> int | None:
    candidate_urls = {requested_url.rstrip("/")}
    if current_url:
        candidate_urls.add(current_url.rstrip("/"))

    status: int | None = None
    # Process each available item while preserving the current workflow state.
    for entry in log_entries:
        # Handle expected failures with a clear, actionable message.
        try:
            message = json.loads(entry["message"])["message"]
            if message.get("method") != "Network.responseReceived":
                continue
            params = message.get("params", {})
            response = params.get("response", {})
            response_url = str(response.get("url", "")).rstrip("/")
            if params.get("type") != "Document" or response_url not in candidate_urls:
                continue
            status = int(float(response["status"]))
        except (KeyError, TypeError, ValueError, json.JSONDecodeError):
            continue
    return status


In [ ]:
# Retrieve the selected match through the exact SofaScore endpoint.
def payload_reports_404(payload: dict[str, Any]) -> bool:
    values: list[Any] = [
        payload.get("status"),
        payload.get("statusCode"),
        payload.get("code"),
    ]
    error_value = payload.get("error")
    # Choose the appropriate path for the current data state.
    if isinstance(error_value, dict):
        values.extend(
            [
                error_value.get("status"),
                error_value.get("statusCode"),
                error_value.get("code"),
            ]
        )
    elif isinstance(error_value, str) and "404" in error_value:
        return True
    return any(str(value).strip() == "404" for value in values if value is not None)


# Handle odds json for reuse in the workflow.
def retrieve_odds_json(selected_match_id: int) -> dict[str, Any]:
    url = (
        f"https://www.sofascore.com/api/v1/"
        f"event/{selected_match_id}/odds/1/all"
    )
    print(f"Opening: {url}")
    clear_performance_log(driver)

    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
    except TimeoutException as exc:
        status = find_document_status(
            read_performance_log(driver), url, safe_current_url(driver)
        )
        # Validate the input before continuing with later processing.
        if status == 404:
            raise OddsNotFoundError(
                "SofaScore returned HTTP 404; no odds are available for this match."
            ) from exc
        raise OddsRetrievalError(
            f"Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading the endpoint."
        ) from exc
    except WebDriverException as exc:
        raise OddsRetrievalError(f"Chrome could not load the endpoint: {exc}") from exc

    status = find_document_status(
        read_performance_log(driver), url, safe_current_url(driver)
    )
    # Validate the input before continuing with later processing.
    if status == 404:
        raise OddsNotFoundError(
            "SofaScore returned HTTP 404; no odds are available for this match."
        )
    # Validate the input before continuing with later processing.
    if status is not None and status >= 400:
        raise OddsRetrievalError(f"SofaScore returned HTTP {status}.")

    # Handle empty body text for reuse in the workflow.
    def non_empty_body_text(current_driver: Any) -> str | bool:
        body_text = current_driver.find_element(By.TAG_NAME, "body").text.strip()
        return body_text if body_text else False

    # Handle expected failures with a clear, actionable message.
    try:
        response_text = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
            non_empty_body_text
        )
    except TimeoutException as exc:
        raise OddsRetrievalError(
            f"No readable response body appeared within {WAIT_TIMEOUT_SECONDS} seconds."
        ) from exc
    except WebDriverException as exc:
        raise OddsRetrievalError(f"Chrome could not read the response body: {exc}") from exc

    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise InvalidOddsJSONError(
            f"SofaScore did not return valid JSON (line {exc.lineno}, column {exc.colno})."
        ) from exc
    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise InvalidOddsJSONError("The SofaScore response must be a JSON object.")
    # Validate the input before continuing with later processing.
    if payload_reports_404(payload):
        raise OddsNotFoundError(
            "SofaScore returned HTTP 404; no odds are available for this match."
        )
    return payload


In [ ]:
# Extract current full-time 1X2 decimal odds.
def extract_full_time_1x2(
    payload: dict[str, Any],
) -> tuple[list[dict[str, Any]], list[str]]:
    markets = payload.get("markets")
    # Validate the input before continuing with later processing.
    if not isinstance(markets, list):
        raise OddsUnavailableError("The response does not contain a markets list.")

    bookmakers: list[dict[str, Any]] = []
    issues: list[str] = []
    qualifying_count = 0
    required_names = {"1", "X", "2"}

    # Process each available item while preserving the current workflow state.
    for index, market in enumerate(markets, start=1):
        if not isinstance(market, dict):
            continue
        if not (
            market.get("marketId") == 1
            and market.get("marketGroup") == "1X2"
            and market.get("marketPeriod") == "Full-time"
        ):
            continue

        qualifying_count += 1
        source_id = market.get("sourceId")
        label = f"market {index} (sourceId={source_id!r})"
        if market.get("suspended") is True:
            issues.append(f"Skipped suspended {label}.")
            continue
        if source_id is None:
            issues.append(f"Skipped {label}: sourceId is missing.")
            continue

        choices = market.get("choices")
        if not isinstance(choices, list):
            issues.append(f"Skipped {label}: choices is not a list.")
            continue
        outcomes = {
            choice.get("name"): choice
            for choice in choices
            if isinstance(choice, dict) and choice.get("name") in required_names
        }
        missing = required_names - outcomes.keys()
        if missing:
            issues.append(f"Skipped {label}: missing {', '.join(sorted(missing))}.")
            continue

        # Handle expected failures with a clear, actionable message.
        try:
            bookmakers.append(
                {
                    "bookmaker": None,
                    "bookmaker_source_id": source_id,
                    "home_win": fractional_to_decimal(
                        outcomes["1"].get("fractionalValue")
                    ),
                    "draw": fractional_to_decimal(
                        outcomes["X"].get("fractionalValue")
                    ),
                    "away_win": fractional_to_decimal(
                        outcomes["2"].get("fractionalValue")
                    ),
                }
            )
        except ValueError as exc:
            issues.append(f"Skipped {label}: {exc}")

    # Validate the input before continuing with later processing.
    if qualifying_count == 0:
        raise OddsUnavailableError("No full-time 1X2 market is available.")
    if not bookmakers and not issues:
        issues.append("No valid full-time 1X2 odds could be extracted.")
    return bookmakers, issues


In [ ]:
# Retrieve the chosen match and print the result in the console.
result: dict[str, Any] = {
    "match_id": match_id,
    "bookmakers": [],
    "issues": [],
}

# Handle expected failures with a clear, actionable message.
try:
    payload = retrieve_odds_json(match_id)
    bookmakers, extraction_issues = extract_full_time_1x2(payload)
    result["bookmakers"] = bookmakers
    result["issues"].extend(extraction_issues)
except OddsNotFoundError as exc:
    result["issues"].append(str(exc))
except OddsTestError as exc:
    result["issues"].append(f"{type(exc).__name__}: {exc}")
except Exception as exc:
    result["issues"].append(f"Unexpected error: {type(exc).__name__}: {exc}")

print("\nStructured result:")
print(json.dumps(result, ensure_ascii=False, indent=2))


In [ ]:
# Close Chrome after inspecting the result.
if driver is not None:
    # Handle expected failures with a clear, actionable message.
    try:
        driver.quit()
        print("Chrome driver closed.")
    except Exception as exc:
        print(f"Chrome driver shutdown warning: {exc}")
    finally:
        driver = None
else:
    print("Chrome driver is already closed or was not initialized.")
